# Step 1: Load and Inspect Data

In [41]:
!pip install dash

In [42]:
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px
import plotly.graph_objs as go
import pandas as pd
import numpy as np
import seaborn as sns
from dash import Dash

In [43]:
df = pd.read_csv('https://confrecordings.ams3.cdn.digitaloceanspaces.com/exp3ch10%2FCAR_market_data1.csv')
df.drop(df.columns[[0]], axis=1, inplace=True)
df.dropna(inplace=True)

In [44]:
maker_wise_df = (df['Make'].value_counts())
pie_carmaker_wise = px.pie(maker_wise_df, values=maker_wise_df.values, names=maker_wise_df.index, color_discrete_sequence=px.colors.sequential.Reds_r)
pie_carmaker_wise.update_traces(textposition='inside')
pie_carmaker_wise.update_layout(uniformtext_minsize=12, uniformtext_mode='hide')
pie_carmaker_wise.update_layout(title_text="All Manufactures", title_x=0.5)

# Step 2: Create App Structure

In [45]:
app = dash.Dash(__name__)
app.layout = html.Div(children=[

    #This is the main header of the dashboard displaying the name - car market DASHBOARD
    html.Div(children=[

            html.H1(children='CAR MARKET DASHBOARD'),
            html.Div(children='A one-stop dashboard to get all your information about cars')],
            style={'textAlign': 'center','backgroundColor':'#E23744','color': 'black','font-family':['Open Sans','sans-serif'],
                   'font-style': ['italic'],'padding-top':'20px','padding-bottom':'40px','fontSize':19}
            ),

    #This is the first row of the dashboard displaying three cards
    html.Div(children=[

            html.Div([
                html.H3(children="Number of total cars in market", style={'fontSize':25}),
                html.P(df.shape[0], style={'fontSize':30})],
                style={'display':'inline-block','height':'400px','width': '28%','textAlign': 'center','backgroundColor': '#2D2D2D',
                       'color': 'white','margin':'25px','border-radius':'5px','box-shadow':'2px 2px 2px #1f2c56'}),


            html.Div([
                html.H3(children="Number of total car models by maker", style={'fontSize':25}),
                html.P(id="numOfmakers", children=22, style={'fontSize':30})],
                style={'display':'inline-block','height':'400px','width': '28%','textAlign': 'center','backgroundColor': '#2D2D2D',
                       'color': 'white','margin':'25px','border-radius':'5px','box-shadow':'2px 2px 2px #1f2c56'}),


            html.Div([
                html.H3(children="number of total variants by car model", style={'fontSize':25}),
                html.P(id="numOfmodels",children=6, style={'fontSize':30})],
                style={'display':'inline-block','height':'400px','width': '28%','textAlign': 'center','backgroundColor': '#2D2D2D',
                       'color': 'white','margin':'25px','border-radius':'5px','box-shadow':'2px 2px 2px #1f2c56'}),

        ]),

    #This is the second row of the dashboard
    html.Div(children=[

            #This first div in the second row contains three different Dash core components
            #Two dropdown lists and a slider
            html.Div(children=[

                #The first component in this Div is a dropdown menu
                html.P('Select Maker: ', style={'color':'white'}),
                dcc.Dropdown(
                        id="cars_dropdown",
                        multi=False,
                        clearable=True,
                        value='Tata',
                        placeholder="Select Maker:",
                        options=[{'label':c, 'value':c} for c in (df['Make'].unique())]),
                html.Br(),
                html.Br(),

                #The second component in this Div is another dropdown menu
                html.P('Select Model: ', style={'color':'white'}),
                dcc.Dropdown(
                        id="models_dropdown",
                        multi=False,
                        clearable=True,
                        value='Nexon',
                        placeholder="Select Model:",
                        options=[]),
                html.Br(),
                html.Br(),



                ],
                style={'display':'inline-block','textAlign': 'left','backgroundColor': '#2D2D2D','color': 'black',
                        'margin-left':'25px','margin-right':'25px','height':'400px','width':'30%','border-radius':'5px',
                        'box-shadow':'2px 2px 2px #1f2c56','padding':'25px'}
            ),


            #The second div in the second row displays a static pie chart
            html.Div([
                    dcc.Graph(
                            id="pie-chart1", figure=pie_carmaker_wise,
                            style={'display':'inline-block','height':'500px','width':'57vh',
                                    'margin-left':'25px','margin-right':'25px','align':'center'})
                    ]),

            #This third div in the second row displays a bar chart

            html.Div([
                    dcc.Graph(
                            id="bar-chart",
                            style={'display':'inline-block','height':'500px','width':'57vh','margin-left':'25px','margin-right':'25px',
                                   'align':'center'})]
                    )],

            style={'display':'flex'}
        ),

    #This is the third row of the dashboard
    html.Div([

            #The first div in this row displays a grouped bar chart

            html.Div([
                    dcc.Graph(
                            id="grouped-bar-chart",
                            style={'display':'inline-block','height':'400px','width':'57vh','margin-left':'25px','margin-right':'25px'})
                    ]),


            #The second div in this row displays a scatter plot

            html.Div([
                    dcc.Graph(
                            id="scatter_plot",
                            style={'display':'inline-block','height':'400px','width':'62vh','margin-left':'25px','margin-right':'1px'})
                    ]),


            #The third div in this row displays a donut chart

            html.Div([
                    dcc.Graph(
                            id="donut_graph",
                            style={'display':'inline-block','height':'400px','width':'57vh','margin-left':'25px','margin-right':'25px'})]
                    )],

            style={'display':'flex','margin-top':'25px'}
        ),


    #This is the fourth row of the dashboard
    html.Div([


        html.Div([dcc.Graph(id="world_map")],style={'width':'90%','align':'center','margin-left':'25px','margin-right':'25px'})


        ]),
    #This is the footer of the dashboard
    html.Div(children=[

            html.Div(children='')],
            style={'textAlign': 'center','backgroundColor':'#E23744','color': 'white','font-family':['Open Sans','sans-serif'],
                   'font-style': ['italic'],'padding-top':'20px','padding-bottom':'20px','fontSize':17}
            )

])

# Step 3: Create Callback Functions

In [46]:
# dropdown menu for makers
@app.callback(
    Output("models_dropdown", "options"),
    [Input("cars_dropdown", "value")])
def get_models_options(cars):
    df_result = df[df['Make']==cars]
    return [{'label':i , 'value': i} for i in df_result['Make']]

In [47]:
#dropdown menu for models
@app.callback(
    Output("models_dropdown", "value"),
    [Input("models_dropdown", "options")])
def get_models_options(models):
    car_df=df[df['Model']==models]
    df_result = car_df['Model'].unique()
    return df_result

In [48]:
#Number of models
@app.callback(
    Output("numOfmakers", "children"),
    [Input("cars_dropdown", "value")])
def get_models_options(cars_dropdown):
    df_result = df[df['Make']==cars_dropdown]
    return df_result.shape[0]

In [49]:
# number of variants
@app.callback(
    Output("numOfmodels", "children"),
    [Input("cars_dropdown", "value")])
def get_models_options(models):
    car_df=df[df['Make']==models]
    df_result = car_df['Model'].value_counts()[0]
    return df_result.shape[0]

In [50]:
#barchart
@app.callback(
    Output("bar-chart", "figure"),
    [Input("cars_dropdown", "value")])
def update_bar_chart(cars):
    car_wise_df = df[df['Make']==cars]
    models_count = (car_wise_df['Model'].value_counts())
    fig = px.bar(models_count, y=models_count.index[:10], x=models_count.values[:10],
                 labels={"x": "cars","y": "Number of cars"},color_discrete_sequence=px.colors.qualitative.Set1)
    fig.update_layout(plot_bgcolor="#f4f4f2")
    fig.update_layout(title_text='Top  models in Selected manufactures', title_x=0.5)
    return fig

In [51]:
#grouped bar chart
@app.callback(
    Output("grouped-bar-chart", "figure"),
    [Input("cars_dropdown", "value")])
def update_grouped_bar_chart(cars):
    car_wise_df = df[df['Make']==cars]
    models_count = (car_wise_df['Make'].value_counts())
    top_10_models = list(models_count.index[:10])

    top_10_models_df = car_wise_df[car_wise_df["Mole"].isin(top_10_models)]

    fig2=px.histogram(top_10_models_df, y=top_10_models_df['Model'], color="Fuel_Type",barmode='group',
                      color_discrete_sequence=px.colors.sequential.Reds_r)
    fig2.update_layout(plot_bgcolor="#f4f4f2")
    fig2.update_layout(title_text='car types', title_x=0.5)

    return fig2

In [52]:
# scatterplot
@app.callback(
    Output("scatter_plot", "figure"),
    [Input("cars_dropdown", "value")])
def update_scatter_plot(models):
    model_wise_df = df[df['Make']==models]
    fig = px.scatter(model_wise_df, x="Ex-Showroom_Price", y="Body_Type",color="Ex-Showroom_Price",
                     color_continuous_scale=px.colors.sequential.Reds_r,hover_data=["Model"])
    fig.update_layout(plot_bgcolor="#f4f4f2")
    fig.update_layout(title_text='car price vs body type', title_x=0.2)
    return fig

In [53]:
#donut graph
@app.callback(
    Output("donut_graph", "figure"),
    [Input("cars_dropdown", "value")])
def update_scatter_plot(cars):
    car_wise_df = df[df['Make']==cars]
    type_wise_df = car_wise_df['Type'].value_counts()
    pie_type_wise = px.pie(type_wise_df, values=type_wise_df.values, names=type_wise_df.index,
                             color_discrete_sequence=px.colors.sequential.Reds_r, hole=0.6)
    pie_type_wise.update_traces(textposition='inside')
    pie_type_wise.update_layout(uniformtext_minsize=12, uniformtext_mode='hide')
    pie_type_wise.update_layout(title_text="type of cars")
    return pie_type_wise

In [54]:
#worldmap
@app.callback(
    Output("world_map", "figure"),
    [Input("cars_dropdown", "value")])
def update_world_map(val):
    df1=df[df['Make']==val]
    fig = px.scatter_mapbox(df1, lat="lat", lon="lng", hover_name="city",hover_data=["city", "Model"], template='plotly_dark', title="different car market acrross india", color_discrete_sequence=["fuchsia"], zoom=3)
    fig.update_layout(mapbox_style="open-street-map")
    return fig

In [55]:
!pip install flask-ngrok

In [64]:
# Change debug to False to avoid the SystemExit conflict in notebooks
from google.colab import output
output.serve_kernel_port_as_iframe(8051)

if __name__ == '__main__':
    app.run(debug=False, port=8051)

<IPython.core.display.Javascript object>

Dash is running on http://127.0.0.1:8051/



INFO:dash.dash:Dash is running on http://127.0.0.1:8051/



 * Serving Flask app '__main__'
 * Debug mode: off


/tmp/ipython-input-49-3038869069.py:7: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

